# Clases y objetos · Estructuras lineales

Listas enlazadas, pilas, colas, tablas hash y caché LRU implementadas como clases. En una lista enlazada insertar al principio cuesta $O(1)$ y buscar $O(n)$; apilar, desapilar, encolar y desencolar cuestan $O(1)$; en una tabla hash `put`, `get` y `delete` cuestan $O(1)$ de media.

## 1. Lista enlazada simple

Definir `Nodo` y `ListaEnlazada` con `insertar_inicio`, `insertar_final` (ambas $O(1)$ guardando un puntero al último nodo), `buscar` (posición del valor o `-1`), `borrar` (primera aparición; devuelve si lo encontró), `__len__`, `__iter__` y `__repr__`.

In [1]:
class Nodo:
    def __init__(self, valor, siguiente=None):
        self.valor = valor
        self.siguiente = siguiente


class ListaEnlazada:
    def __init__(self, valores=()):
        self.cabeza = self.cola = None
        self.n = 0
        for v in valores:
            self.insertar_final(v)

    def insertar_inicio(self, valor):
        self.cabeza = Nodo(valor, self.cabeza)
        if self.cola is None:
            self.cola = self.cabeza
        self.n += 1

    def insertar_final(self, valor):
        nodo = Nodo(valor)
        if self.cola is None:
            self.cabeza = nodo
        else:
            self.cola.siguiente = nodo
        self.cola = nodo
        self.n += 1

    def buscar(self, valor):
        for i, v in enumerate(self):
            if v == valor:
                return i
        return -1

    def borrar(self, valor):
        anterior, actual = None, self.cabeza
        while actual and actual.valor != valor:
            anterior, actual = actual, actual.siguiente
        if actual is None:
            return False
        if anterior is None:
            self.cabeza = actual.siguiente
        else:
            anterior.siguiente = actual.siguiente
        if actual is self.cola:
            self.cola = anterior
        self.n -= 1
        return True

    def __len__(self):
        return self.n

    def __iter__(self):
        actual = self.cabeza
        while actual:
            yield actual.valor
            actual = actual.siguiente

    def __repr__(self):
        return " -> ".join([*map(repr, self), "None"])


lista = ListaEnlazada([3, 1, 4])
lista.insertar_inicio(9)
lista.insertar_final(5)
print(lista, len(lista))
print(lista.buscar(4), lista.buscar(7))
print(lista.borrar(9), lista.borrar(5), lista.borrar(7))
print(lista, len(lista))

9 -> 3 -> 1 -> 4 -> 5 -> None 5
3 -1
True True False
3 -> 1 -> 4 -> None 3


## 2. Invertir y comprobar contra `list`

Ampliar la clase (heredando de la anterior) con `invertir`, que da la vuelta a la lista cambiando los enlaces, sin crear nodos, en $O(n)$. Comprobar todas las operaciones contra una `list` con operaciones aleatorias.

In [2]:
import random


class ListaEnlazada(ListaEnlazada):
    def invertir(self):
        anterior, actual = None, self.cabeza
        self.cola = actual
        while actual:
            siguiente = actual.siguiente
            actual.siguiente = anterior
            anterior, actual = actual, siguiente
        self.cabeza = anterior


lista = ListaEnlazada("abcde")
lista.invertir()
print(lista)

random.seed(1)
for _ in range(300):
    lista, modelo = ListaEnlazada(), []
    for _ in range(30):
        op, x = random.choice("ifbsv"), random.randint(0, 9)
        if op == "i":
            lista.insertar_inicio(x)
            modelo.insert(0, x)
        elif op == "f":
            lista.insertar_final(x)
            modelo.append(x)
        elif op == "b":
            assert lista.borrar(x) == (x in modelo)
            if x in modelo:
                modelo.remove(x)
        elif op == "s":
            assert lista.buscar(x) == (modelo.index(x) if x in modelo else -1)
        else:
            lista.invertir()
            modelo.reverse()
        assert list(lista) == modelo and len(lista) == len(modelo)
        assert (lista.cola.valor if lista.cola else None) == (modelo[-1] if modelo else None)
print("Comprobación correcta")

'e' -> 'd' -> 'c' -> 'b' -> 'a' -> None
Comprobación correcta


## 3. Lista doblemente enlazada

Cada nodo enlaza con el anterior y con el siguiente. Implementar `insertar_inicio`, `insertar_final`, `quitar_inicio` y `quitar_final` (todas $O(1)$), `borrar` un valor y `__reversed__` para recorrerla al revés. Comprobar contra `collections.deque`.

In [3]:
from collections import deque


class NodoDoble:
    def __init__(self, valor):
        self.valor = valor
        self.anterior = self.siguiente = None


class ListaDoble:
    def __init__(self, valores=()):
        self.cabeza = self.cola = None
        self.n = 0
        for v in valores:
            self.insertar_final(v)

    def insertar_inicio(self, valor):
        nodo = NodoDoble(valor)
        nodo.siguiente = self.cabeza
        if self.cabeza:
            self.cabeza.anterior = nodo
        else:
            self.cola = nodo
        self.cabeza = nodo
        self.n += 1

    def insertar_final(self, valor):
        nodo = NodoDoble(valor)
        nodo.anterior = self.cola
        if self.cola:
            self.cola.siguiente = nodo
        else:
            self.cabeza = nodo
        self.cola = nodo
        self.n += 1

    def _desenlazar(self, nodo):
        if nodo.anterior:
            nodo.anterior.siguiente = nodo.siguiente
        else:
            self.cabeza = nodo.siguiente
        if nodo.siguiente:
            nodo.siguiente.anterior = nodo.anterior
        else:
            self.cola = nodo.anterior
        self.n -= 1
        return nodo.valor

    def quitar_inicio(self):
        if self.cabeza is None:
            raise IndexError("lista vacía")
        return self._desenlazar(self.cabeza)

    def quitar_final(self):
        if self.cola is None:
            raise IndexError("lista vacía")
        return self._desenlazar(self.cola)

    def borrar(self, valor):
        nodo = self.cabeza
        while nodo:
            if nodo.valor == valor:
                self._desenlazar(nodo)
                return True
            nodo = nodo.siguiente
        return False

    def __len__(self):
        return self.n

    def __iter__(self):
        nodo = self.cabeza
        while nodo:
            yield nodo.valor
            nodo = nodo.siguiente

    def __reversed__(self):
        nodo = self.cola
        while nodo:
            yield nodo.valor
            nodo = nodo.anterior

    def __repr__(self):
        return "None <-> " + " <-> ".join([*map(repr, self), "None"])


doble = ListaDoble([1, 2, 3])
doble.insertar_inicio(0)
print(doble, list(reversed(doble)))
print(doble.quitar_final(), doble.quitar_inicio(), doble)
try:
    ListaDoble().quitar_inicio()
except IndexError as e:
    print("Error:", e)

random.seed(2)
for _ in range(300):
    doble, modelo = ListaDoble(), deque()
    for _ in range(30):
        op, x = random.choice("ifIFb"), random.randint(0, 9)
        if op == "i":
            doble.insertar_inicio(x)
            modelo.appendleft(x)
        elif op == "f":
            doble.insertar_final(x)
            modelo.append(x)
        elif op == "I" and modelo:
            assert doble.quitar_inicio() == modelo.popleft()
        elif op == "F" and modelo:
            assert doble.quitar_final() == modelo.pop()
        elif op == "b":
            assert doble.borrar(x) == (x in modelo)
            if x in modelo:
                modelo.remove(x)
        assert list(doble) == list(modelo) and list(reversed(doble)) == list(reversed(modelo))
print("Comprobación correcta")

None <-> 0 <-> 1 <-> 2 <-> 3 <-> None [3, 2, 1, 0]
3 0 None <-> 1 <-> 2 <-> None
Error: lista vacía
Comprobación correcta


## 4. Pila

Clase `Pila` sobre una lista de Python (el final de la lista es la cima): `apilar`, `desapilar`, `cima`, `esta_vacia` y `__len__`. Desapilar o consultar una pila vacía lanza `IndexError`.

In [4]:
class Pila:
    def __init__(self):
        self.datos = []

    def apilar(self, valor):
        self.datos.append(valor)

    def desapilar(self):
        if not self.datos:
            raise IndexError("pila vacía")
        return self.datos.pop()

    def cima(self):
        if not self.datos:
            raise IndexError("pila vacía")
        return self.datos[-1]

    def esta_vacia(self):
        return not self.datos

    def __len__(self):
        return len(self.datos)

    def __repr__(self):
        return f"Pila({self.datos})"


pila = Pila()
for x in "abc":
    pila.apilar(x)
print(pila, pila.cima(), len(pila))
print([pila.desapilar() for _ in range(3)], pila.esta_vacia())
try:
    pila.desapilar()
except IndexError as e:
    print("Error:", e)

Pila(['a', 'b', 'c']) c 3
['c', 'b', 'a'] True
Error: pila vacía


## 5. Paréntesis balanceados

Usar la pila para comprobar si los `()`, `[]` y `{}` de una expresión están bien emparejados y anidados: cada apertura se apila y cada cierre debe coincidir con la cima. Al final la pila debe quedar vacía.

In [5]:
PAREJAS = {")": "(", "]": "[", "}": "{"}


def balanceados(texto):
    pila = Pila()
    for c in texto:
        if c in "([{":
            pila.apilar(c)
        elif c in PAREJAS:
            if pila.esta_vacia() or pila.desapilar() != PAREJAS[c]:
                return False
    return pila.esta_vacia()


for expr in ["(a + b) * [c - {d / e}]", "([)]", "((x)", "x))", "", "{[()()]}"]:
    print(f"{expr!r:28} {balanceados(expr)}")

assert balanceados("{[()()]}") and not balanceados("([)]") and not balanceados("((")

'(a + b) * [c - {d / e}]'    True
'([)]'                       False
'((x)'                       False
'x))'                        False
''                           True
'{[()()]}'                   True


## 6. Notación polaca inversa

Evaluar expresiones en notación polaca inversa (postfija), como `3 4 + 2 *` $= 14$: los números se apilan y cada operador desapila dos operandos y apila el resultado. Dar un error si faltan o sobran operandos.

In [6]:
import operator

OPERADORES = {"+": operator.add, "-": operator.sub, "*": operator.mul, "/": operator.truediv, "^": operator.pow}


def evaluar_rpn(expresion):
    pila = Pila()
    for token in expresion.split():
        if token in OPERADORES:
            if len(pila) < 2:
                raise ValueError(f"faltan operandos para '{token}'")
            b, a = pila.desapilar(), pila.desapilar()
            pila.apilar(OPERADORES[token](a, b))
        else:
            pila.apilar(float(token))
    if len(pila) != 1:
        raise ValueError("sobran operandos")
    return pila.desapilar()


for expr in ["3 4 + 2 *", "5 1 2 + 4 * + 3 -", "2 3 ^ 1 -", "10 4 /", "3 +", "1 2 3 +"]:
    try:
        print(f"{expr:20} = {evaluar_rpn(expr)}")
    except ValueError as e:
        print(f"{expr:20} -> Error: {e}")

assert evaluar_rpn("5 1 2 + 4 * + 3 -") == 5 + (1 + 2) * 4 - 3
assert evaluar_rpn("2 3 4 * +") == 2 + 3 * 4

3 4 + 2 *            = 14.0
5 1 2 + 4 * + 3 -    = 14.0
2 3 ^ 1 -            = 7.0
10 4 /               = 2.5
3 +                  -> Error: faltan operandos para '+'
1 2 3 +              -> Error: sobran operandos


## 7. Cola con lista enlazada y con dos pilas

Implementar `Cola` con nodos enlazados (punteros al frente y al final) y `ColaDosPilas` con una pila de entrada y otra de salida: cuando la de salida está vacía se vuelca en ella la de entrada, así cada elemento cambia de pila una sola vez y desencolar cuesta $O(1)$ amortizado. Comprobar ambas contra `collections.deque` y comparar tiempos.

In [7]:
import time


class Cola:
    def __init__(self):
        self.frente = self.final = None
        self.n = 0

    def encolar(self, valor):
        nodo = Nodo(valor)
        if self.final:
            self.final.siguiente = nodo
        else:
            self.frente = nodo
        self.final = nodo
        self.n += 1

    def desencolar(self):
        if self.frente is None:
            raise IndexError("cola vacía")
        valor = self.frente.valor
        self.frente = self.frente.siguiente
        if self.frente is None:
            self.final = None
        self.n -= 1
        return valor

    def __len__(self):
        return self.n


class ColaDosPilas:
    def __init__(self):
        self.entrada, self.salida = Pila(), Pila()

    def encolar(self, valor):
        self.entrada.apilar(valor)

    def desencolar(self):
        if self.salida.esta_vacia():
            while not self.entrada.esta_vacia():
                self.salida.apilar(self.entrada.desapilar())
        return self.salida.desapilar()

    def __len__(self):
        return len(self.entrada) + len(self.salida)


random.seed(3)
for clase in (Cola, ColaDosPilas):
    cola, modelo = clase(), deque()
    for _ in range(5000):
        if random.random() < 0.55:
            x = random.randint(0, 99)
            cola.encolar(x)
            modelo.append(x)
        elif modelo:
            assert cola.desencolar() == modelo.popleft()
        assert len(cola) == len(modelo)
print("Comprobación correcta")


def medir(cola, encolar, desencolar, n=100_000):
    inicio = time.perf_counter()
    for i in range(n):
        encolar(cola, i)
    for _ in range(n):
        desencolar(cola)
    return time.perf_counter() - inicio


for nombre, cola, enc, des in [("Cola", Cola(), Cola.encolar, Cola.desencolar),
                               ("ColaDosPilas", ColaDosPilas(), ColaDosPilas.encolar, ColaDosPilas.desencolar),
                               ("deque", deque(), deque.append, deque.popleft)]:
    print(f"{nombre:13} {medir(cola, enc, des):.3f} s")

Comprobación correcta
Cola          0.024 s
ColaDosPilas  0.021 s
deque         0.003 s


## 8. Cola circular de tamaño fijo

Cola de capacidad fija sobre una lista de tamaño `capacidad`: `inicio` señala el primer elemento y la posición del siguiente hueco se calcula con módulo, así nunca hay que mover elementos. Encolar en una cola llena lanza `OverflowError`.

In [8]:
class ColaCircular:
    def __init__(self, capacidad):
        self.datos = [None] * capacidad
        self.inicio = 0
        self.n = 0

    def esta_llena(self):
        return self.n == len(self.datos)

    def encolar(self, valor):
        if self.esta_llena():
            raise OverflowError("cola llena")
        self.datos[(self.inicio + self.n) % len(self.datos)] = valor
        self.n += 1

    def desencolar(self):
        if self.n == 0:
            raise IndexError("cola vacía")
        valor = self.datos[self.inicio]
        self.datos[self.inicio] = None
        self.inicio = (self.inicio + 1) % len(self.datos)
        self.n -= 1
        return valor

    def __len__(self):
        return self.n

    def __iter__(self):
        for i in range(self.n):
            yield self.datos[(self.inicio + i) % len(self.datos)]

    def __repr__(self):
        return f"ColaCircular({list(self)}, capacidad={len(self.datos)})"


cola = ColaCircular(3)
for x in (1, 2, 3):
    cola.encolar(x)
try:
    cola.encolar(4)
except OverflowError as e:
    print("Error:", e)
print(cola.desencolar(), cola.desencolar())
cola.encolar(4)
cola.encolar(5)
print(cola, "| interna:", cola.datos, "| inicio:", cola.inicio)

random.seed(4)
cola, modelo = ColaCircular(5), deque()
for _ in range(5000):
    if random.random() < 0.5:
        x = random.randint(0, 99)
        if len(modelo) == 5:
            try:
                cola.encolar(x)
                raise AssertionError("debía estar llena")
            except OverflowError:
                pass
        else:
            cola.encolar(x)
            modelo.append(x)
    elif modelo:
        assert cola.desencolar() == modelo.popleft()
    assert list(cola) == list(modelo)
print("Comprobación correcta")

Error: cola llena
1 2
ColaCircular([3, 4, 5], capacidad=3) | interna: [4, 5, 3] | inicio: 2
Comprobación correcta


## 9. Tabla hash con encadenamiento

Cada cubeta es una lista de pares `[clave, valor]` y la cubeta de una clave es `hash(clave) % m`. Implementar `put`, `get` y `delete`, más `__setitem__`, `__getitem__`, `__delitem__`, `__contains__`, `__len__` e `__iter__`. Cuando el factor de carga $n / m$ supera $0.75$ se duplica el número de cubetas $m$ y se reinsertan todas las claves. Comprobar contra `dict`.

In [9]:
class TablaHash:
    def __init__(self, capacidad=8, carga_maxima=0.75):
        self.cubetas = [[] for _ in range(capacidad)]
        self.n = 0
        self.carga_maxima = carga_maxima

    def _cubeta(self, clave):
        return self.cubetas[hash(clave) % len(self.cubetas)]

    def _par(self, clave):
        for par in self._cubeta(clave):
            if par[0] == clave:
                return par
        return None

    def put(self, clave, valor):
        par = self._par(clave)
        if par:
            par[1] = valor
            return
        self._cubeta(clave).append([clave, valor])
        self.n += 1
        if self.n / len(self.cubetas) > self.carga_maxima:
            self._redimensionar(2 * len(self.cubetas))

    def get(self, clave, defecto=None):
        par = self._par(clave)
        return defecto if par is None else par[1]

    def delete(self, clave):
        cubeta = self._cubeta(clave)
        for i, par in enumerate(cubeta):
            if par[0] == clave:
                del cubeta[i]
                self.n -= 1
                return True
        return False

    def _redimensionar(self, capacidad):
        pares = list(self.items())
        self.cubetas = [[] for _ in range(capacidad)]
        for clave, valor in pares:
            self._cubeta(clave).append([clave, valor])

    def items(self):
        for cubeta in self.cubetas:
            for clave, valor in cubeta:
                yield clave, valor

    def __getitem__(self, clave):
        par = self._par(clave)
        if par is None:
            raise KeyError(clave)
        return par[1]

    def __setitem__(self, clave, valor):
        self.put(clave, valor)

    def __delitem__(self, clave):
        if not self.delete(clave):
            raise KeyError(clave)

    def __contains__(self, clave):
        return self._par(clave) is not None

    def __len__(self):
        return self.n

    def __iter__(self):
        return (clave for clave, _ in self.items())

    def __repr__(self):
        return f"TablaHash({self.n} claves, {len(self.cubetas)} cubetas)"


tabla = TablaHash()
m = len(tabla.cubetas)
for palabra in "el perro de san roque no tiene rabo porque ramón ramírez se lo ha cortado".split():
    tabla[palabra] = tabla.get(palabra, 0) + 1
    if len(tabla.cubetas) != m:
        m = len(tabla.cubetas)
        print(f"Redimensionada a {m} cubetas con {len(tabla)} claves")
print(tabla, tabla["ramón"], "perro" in tabla, "gato" in tabla)
print("Cubeta más larga:", max(map(len, tabla.cubetas)))
del tabla["perro"]
try:
    tabla["perro"]
except KeyError as e:
    print("Error: no existe", e)

random.seed(5)
tabla, modelo = TablaHash(), {}
for _ in range(20_000):
    op, clave = random.choice("pgd"), random.choice([random.randint(0, 500), f"k{random.randint(0, 500)}"])
    if op == "p":
        tabla[clave] = modelo[clave] = random.random()
    elif op == "g":
        assert tabla.get(clave) == modelo.get(clave) and (clave in tabla) == (clave in modelo)
    else:
        assert tabla.delete(clave) == (modelo.pop(clave, None) is not None)
    assert len(tabla) == len(modelo)
assert dict(tabla.items()) == modelo and sorted(map(str, tabla)) == sorted(map(str, modelo))
print("Comprobación correcta:", tabla, f"factor de carga {len(tabla) / len(tabla.cubetas):.2f}")

Redimensionada a 16 cubetas con 7 claves
Redimensionada a 32 cubetas con 13 claves
TablaHash(15 claves, 32 cubetas) 1 True False
Cubeta más larga: 2
Error: no existe 'perro'
Comprobación correcta: TablaHash(491 claves, 1024 cubetas) factor de carga 0.48


## 10. Caché LRU

Caché de capacidad fija que, al llenarse, expulsa el elemento usado hace más tiempo (*least recently used*). Un `dict` da acceso $O(1)$ al nodo de cada clave y una lista doblemente enlazada con dos nodos centinela guarda el orden de uso: al leer o escribir una clave su nodo pasa al principio y, si no cabe, se expulsa el del final. Comprobar contra una versión con `OrderedDict`.

In [10]:
from collections import OrderedDict


class NodoLRU:
    def __init__(self, clave=None, valor=None):
        self.clave, self.valor = clave, valor
        self.anterior = self.siguiente = None


class CacheLRU:
    def __init__(self, capacidad):
        self.capacidad = capacidad
        self.nodos = {}
        self.inicio, self.fin = NodoLRU(), NodoLRU()  # centinelas
        self.inicio.siguiente, self.fin.anterior = self.fin, self.inicio

    def _quitar(self, nodo):
        nodo.anterior.siguiente = nodo.siguiente
        nodo.siguiente.anterior = nodo.anterior

    def _poner_delante(self, nodo):
        nodo.anterior, nodo.siguiente = self.inicio, self.inicio.siguiente
        self.inicio.siguiente.anterior = nodo
        self.inicio.siguiente = nodo

    def get(self, clave, defecto=None):
        nodo = self.nodos.get(clave)
        if nodo is None:
            return defecto
        self._quitar(nodo)
        self._poner_delante(nodo)
        return nodo.valor

    def put(self, clave, valor):
        nodo = self.nodos.get(clave)
        if nodo:
            nodo.valor = valor
            self._quitar(nodo)
        else:
            if len(self.nodos) == self.capacidad:
                viejo = self.fin.anterior
                self._quitar(viejo)
                del self.nodos[viejo.clave]
            nodo = self.nodos[clave] = NodoLRU(clave, valor)
        self._poner_delante(nodo)

    def claves(self):
        nodo = self.inicio.siguiente
        while nodo is not self.fin:
            yield nodo.clave
            nodo = nodo.siguiente

    def __len__(self):
        return len(self.nodos)

    def __repr__(self):
        return f"CacheLRU({list(self.claves())})"


cache = CacheLRU(3)
for clave in "abc":
    cache.put(clave, clave.upper())
print(cache)
print(cache.get("a"), cache)
cache.put("d", "D")
print(cache, "| b expulsada:", cache.get("b"))


class LRUReferencia:
    def __init__(self, capacidad):
        self.capacidad, self.datos = capacidad, OrderedDict()

    def get(self, clave, defecto=None):
        if clave not in self.datos:
            return defecto
        self.datos.move_to_end(clave, last=False)
        return self.datos[clave]

    def put(self, clave, valor):
        self.datos[clave] = valor
        self.datos.move_to_end(clave, last=False)
        if len(self.datos) > self.capacidad:
            self.datos.popitem()


random.seed(6)
for capacidad in (1, 2, 5, 10):
    cache, modelo = CacheLRU(capacidad), LRUReferencia(capacidad)
    for _ in range(5000):
        clave = random.randint(0, 15)
        if random.random() < 0.5:
            valor = random.random()
            cache.put(clave, valor)
            modelo.put(clave, valor)
        else:
            assert cache.get(clave) == modelo.get(clave)
        assert list(cache.claves()) == list(modelo.datos)
print("Comprobación correcta")

CacheLRU(['c', 'b', 'a'])
A CacheLRU(['a', 'c', 'b'])
CacheLRU(['d', 'a', 'c']) | b expulsada: None
Comprobación correcta
